# Find a recompile storm with tlparse
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/demos/week5/Recompile_Storm_Demo_Colab.ipynb)

**Dataflow Architectures for AI, Week 5, Module 3: Observability across the stack**

A decode loop compiled with dynamic shapes off gets one compiled plan per input shape. If the attention mask grows by one column every step, every step has a new shape, so every step recompiles. After the recompile limit, Dynamo gives up and runs the function in eager mode. Nothing fails. The run is only slow, and the warning is easy to miss. The lecture calls this a **recompile storm**, and it is one of the Lab 5 scenarios.

`TORCH_TRACE` writes a structured log of every compile, and `tlparse` turns it into a report. Each compile has an ID of the form `frame/compile count`: `[0/0]`, `[0/1]`, `[0/2]`. One frame with many compile counts is a storm. In this notebook you plant a storm in a small decode loop, find it with tlparse, and fix it two ways.

**What you will do**
1. Write a decode step whose cache and mask grow by one token per step.
2. Run it with dynamic shapes off and `TORCH_TRACE` turned on.
3. Read the compile IDs and the per-step times.
4. Fix it with a static cache and a padded mask, and check that it compiles once.
5. Compare with automatic dynamic shapes.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. The notebook takes about three minutes, most of it compiling.

## 1. Install tlparse
tlparse is the PyTorch tool that reads `TORCH_TRACE` logs. The Week 5 provenance demo uses it as well.

In [ ]:
!pip -q install tlparse
import subprocess
import torch
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("tlparse:", subprocess.run(["tlparse", "--version"], capture_output=True, text=True).stdout.strip())

## 2. The decode loop
The script has three versions of the same attention step for one new token:

| Version | Cache | Mask | Compile setting |
|---|---|---|---|
| `storm` | grows with `torch.cat` every step | grows by one column every step | `dynamic=False` |
| `static` | preallocated for 64 tokens, written in place | always 64 columns, future positions set to -1e4 | `dynamic=False` |
| `dynamic` | grows with `torch.cat` | grows | default, automatic dynamic shapes |

The script prints the time of each step and the number of distinct compiled graphs at the end.

In [ ]:
%%writefile /content/storm_demo.py
import sys
import time
import torch
import torch._dynamo

version = sys.argv[1]
torch.manual_seed(0)
dev = "cuda" if torch.cuda.is_available() else "cpu"
D, MAX_LEN, STEPS = 256, 64, 16
wq = torch.randn(D, D, device=dev) / D ** 0.5


def attend(x, k_cache, mask):
    q = x @ wq                                         # [1, D] query for the new token
    scores = (q @ k_cache.T) / D ** 0.5 + mask         # [1, T] one score per cached token
    return torch.softmax(scores, dim=-1) @ k_cache     # [1, D]


step = torch.compile(attend, dynamic=None if version == "dynamic" else False)
x = torch.randn(1, D, device=dev)
k_static = torch.zeros(MAX_LEN, D, device=dev)
k_grow = torch.randn(1, D, device=dev)

for t in range(1, STEPS + 1):
    t0 = time.perf_counter()
    if version == "static":
        k_static[t - 1] = x[0]                         # write the new token into its slot
        mask = torch.full((1, MAX_LEN), -1e4, device=dev)
        mask[:, :t] = 0.0                              # positions after t are padding
        out = step(x, k_static, mask)
    else:
        if t > 1:
            k_grow = torch.cat([k_grow, x], dim=0)     # the cache grows by one row
        mask = torch.zeros(1, k_grow.shape[0], device=dev)   # the mask grows by one column
        out = step(x, k_grow, mask)
    x = out
    if dev == "cuda":
        torch.cuda.synchronize()
    print(f"step {t:2d}: {(time.perf_counter() - t0) * 1e3:8.1f} ms", flush=True)

graphs = torch._dynamo.utils.counters["stats"]["unique_graphs"]
print(f"distinct compiled graphs: {graphs}", flush=True)

## 3. Run the storm with TORCH_TRACE
The cell runs the `storm` version in a separate process with `TORCH_TRACE` turned on, then shows the step times and any warning Dynamo printed.

In [ ]:
import os

def run(version, trace_dir=None):
    env = dict(os.environ)
    if trace_dir:
        subprocess.run(["rm", "-rf", trace_dir])
        env["TORCH_TRACE"] = trace_dir
    p = subprocess.run(["python", "/content/storm_demo.py", version], capture_output=True, text=True, env=env)
    print(p.stdout)
    for line in p.stderr.splitlines():
        if "recompile_limit" in line or "cache_size_limit" in line:
            print("WARNING:", line.strip()[:200])
    return p


p_storm = run("storm", trace_dir="/content/trace_storm")

**Reading the output**
- The first steps each take hundreds of milliseconds or more: every step is a compile.
- Dynamo then prints a warning that it hit the recompile limit, which is 8 by default. From that step on, the times drop to a fraction of a millisecond: the function now runs in eager mode, not compiled.
- The last line counts the distinct compiled graphs, one per shape until the limit.
- Nothing raised an error. Without the step times or the warning, a storm looks like a slow first few seconds.

## 4. Read the compile IDs with tlparse
tlparse writes one folder per compile, named after its ID. The folder `-_0_3_0` is frame 0, compile 3, attempt 0. The cell builds the report and lists the IDs.

In [ ]:
import glob
import re

log = sorted(glob.glob("/content/trace_storm/*.log"))[-1]
subprocess.run(["tlparse", log, "-o", "/content/tl_storm", "--overwrite"], capture_output=True)
ids = []
for path in glob.glob("/content/tl_storm/-_*"):
    m = re.match(r"-_(\d+)_(\d+)_(\d+)$", os.path.basename(path))
    if m:
        ids.append(tuple(int(g) for g in m.groups()))
for frame, compile_n, attempt in sorted(ids):
    print(f"[{frame}/{compile_n}]  attempt {attempt}")
frames = {f for f, _, _ in ids}
print(f"\n{len(ids)} compiles across {len(frames)} frame(s)")

**Reading the output**
- One frame, `0`, with compile counts from `[0/0]` up to the limit. That is the signature of a storm: the same function compiled again and again.
- In the full report, open `index.html` and each compile entry lists the guard that failed. Here it is the size of `k_cache` and `mask`, one row and one column larger than last time.
- Compare with a model that has graph breaks: there you see several frames, each compiled once, such as `[0/0]`, `[1/0]` and `[2/0]`.

## 5. Fix one: a static cache and a padded mask
The fix from the lecture is to make every step the same shape. The cache is allocated once for the maximum length and written in place, and the mask always has the maximum length, with -1e4 at the future positions. A finite value is used, not minus infinity, so the mask is safe in FP16 and on targets without IEEE infinity.

In [ ]:
p_static = run("static")

**Reading the output**
- One compiled graph. The first step pays the compile, and every later step is fast.
- The cost moved: each step now computes scores for all 64 positions, including the padded ones. When you place this kernel on a roofline, count the useful FLOPs, so the padding shows up as a gap. Week 6 returns to this.

## 6. Fix two: let the shapes be dynamic
With the default setting, Dynamo marks the growing dimension as dynamic after the first size mismatch, and compiles one graph that accepts any length.

In [ ]:
p_dynamic = run("dynamic")

**Reading the output**
- Two compiled graphs: one for the first shape, one dynamic graph that every later step reuses.
- This works well on a GPU. A compiled-schedule accelerator such as Spyre plans memory and work division for fixed shapes, so it uses the static fix, or pads to a small set of buckets.

## Try this
1. In the script, change `STEPS` to 6. Is there still a storm? How many compiles do you see?
2. In section 3, set `TORCH_LOGS=recompiles` in the environment as well, and read the guard that failed at each step.
3. Run the `static` version with `TORCH_TRACE` and confirm with tlparse that there is a single `[0/0]`.

## Summary
- With `dynamic=False`, every new shape is a new compile. A mask or cache that grows each step recompiles each step.
- After the recompile limit, Dynamo falls back to eager mode with only a warning.
- `TORCH_TRACE` and tlparse show the storm as one frame with many compile IDs: `[0/0]`, `[0/1]`, `[0/2]` and so on.
- Fix it with static shapes (a preallocated cache and a padded mask) or, on a GPU, with dynamic shapes.

Lecture: Week 5, Module 3, slide 29 (logs recall, reading a tlparse compile ID), and Week 6, Module 3, slides 34 to 36 (the slow decode loop).